In [1]:
from owlready2 import *
from rdflib import Graph

In [2]:
onto = get_ontology("http://example.org/transportation.owl")

with onto:

    class Junction(Thing):
        pass

    class Road(Thing):
        pass

    class Vehicle(Thing):
        pass

    class Signal(Thing):
        pass

    class Incident(Thing):
        pass

print("Ontology classes created successfully!")

Ontology classes created successfully!


In [3]:
with onto:

    class connects(ObjectProperty):
        domain = [Road]
        range = [Junction]

    class hasIncident(ObjectProperty):
        domain = [Road]
        range = [Incident]

    class locatedAt(ObjectProperty):
        domain = [Vehicle]
        range = [Junction]

    class controls(ObjectProperty):
        domain = [Signal]
        range = [Junction]

    class distance(DataProperty, FunctionalProperty):
        domain = [Road]
        range = [float]

    class status(DataProperty, FunctionalProperty):
        domain = [Road]
        range = [str]

    class vehicleType(DataProperty, FunctionalProperty):
        domain = [Vehicle]
        range = [str]

print("Ontology properties created successfully!")

Ontology properties created successfully!


In [4]:
junctions = {}

for name in ["A", "B", "C", "D", "E", "F", "G"]:
    junctions[name] = onto.Junction("Junction_" + name)

roads = [
    ("A", "B", 4),
    ("A", "C", 3),
    ("B", "D", 5),
    ("B", "E", 2),
    ("C", "E", 4),
    ("C", "G", 6),
    ("D", "F", 3),
    ("F", "G", 2)
]

road_instances = {}

for a, b, d in roads:

    road = onto.Road("Road_" + a + "_" + b)

    road.connects = [junctions[a], junctions[b]]
    road.distance = float(d)
    road.status = "Open"

    road_instances[(a, b)] = road

print("Junctions and roads created!")

Junctions and roads created!


In [5]:
ambulance = onto.Vehicle("Ambulance_1")
ambulance.vehicleType = "Ambulance"
ambulance.locatedAt = [junctions["A"]]

signal = onto.Signal("Signal_B")
signal.controls = [junctions["B"]]

print("Vehicle and signal created!")

Vehicle and signal created!


In [6]:
accident = onto.Incident("Accident_1")

road_cg = road_instances[("C", "G")]

road_cg.hasIncident = [accident]
road_cg.status = "Blocked"

print("Incident added to:", road_cg.name)
print("Road Status:", road_cg.status)

Incident added to: Road_C_G
Road Status: Blocked


In [7]:
print("Blocked Roads:")

for road in onto.Road.instances():

    if road.status == "Blocked":
        print(road.name)

Blocked Roads:
Road_C_G


In [8]:
road = road_instances[("C", "G")]

print("Road:", road.name)
print("Connects:")

for junction in road.connects:
    print(junction.name)

Road: Road_C_G
Connects:
Junction_C
Junction_G


In [9]:
from pathlib import Path

project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

output = project_folder / "ontology" / "transportation.owl"
output.parent.mkdir(exist_ok=True)

onto.save(file=str(output), format="rdfxml")

print("Ontology saved successfully!")
print(output)

Ontology saved successfully!
c:\Users\dellu\Desktop\Transportation-Knowledge-Framework\ontology\transportation.owl


In [10]:
g = Graph()
g.parse(str(output), format="xml")

print("Total RDF triples:", len(g))

print("\nSome RDF triples:")

for i, triple in enumerate(g):
    if i == 10:
        break
    print(triple)

Total RDF triples: 107

Some RDF triples:
(rdflib.term.URIRef('http://example.org/transportation.owl#Road_A_B'), rdflib.term.URIRef('http://example.org/transportation.owl#connects'), rdflib.term.URIRef('http://example.org/transportation.owl#Junction_B'))
(rdflib.term.URIRef('http://example.org/transportation.owl#Road'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#Thing'))
(rdflib.term.URIRef('http://example.org/transportation.owl#connects'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#ObjectProperty'))
(rdflib.term.URIRef('http://example.org/transportation.owl#vehicleType'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2002/07/owl#FunctionalProperty'))
(rdflib.term.URIRef('http://example.org/transportation.owl#hasIncident'), rdflib.term.URIRef('http://www.w3.org/2000/01/r

In [11]:
print("Total Classes:", len(list(onto.classes())))
print("Total Junctions:", len(list(onto.Junction.instances())))
print("Total Roads:", len(list(onto.Road.instances())))
print("Total Vehicles:", len(list(onto.Vehicle.instances())))
print("Total Signals:", len(list(onto.Signal.instances())))
print("Total Incidents:", len(list(onto.Incident.instances())))

Total Classes: 5
Total Junctions: 7
Total Roads: 8
Total Vehicles: 1
Total Signals: 1
Total Incidents: 1
